# 01 — Data Loading and Cleaning

Builds `df_development` (2000–2021, used for modelling) and `df_real_world` (2022–2023, unlabeled, kept for reference only) from three sources:

1. **IHME** — mental health disorder prevalence (local CSV, already filtered to EU countries at extraction time)
2. **World Bank API** — socioeconomic indicators
3. **WHO API** — suicide rate (target variable)

The analysis is scoped to the 27 EU member states (`EU_COUNTRIES_ISO` in `src/config.py`). Restricting to the EU keeps the countries reasonably comparable in terms of healthcare systems and statistical reporting standards, which matters both for the cross-country comparisons in `02_eda.ipynb` and for the geographical train/test split used later in `04_models.ipynb`.

EDA starts in `02_eda.ipynb`, once this notebook has produced `data/processed/df_development.parquet` and `data/processed/df_real_world.parquet`.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd

from src import (
    EU_COUNTRIES_ISO,
    WORLD_BANK_INDICATORS,
    WHO_SUICIDE_INDICATOR,
    load_ihme_data,
    fetch_worldbank_indicators,
    fetch_who_suicide_rates,
    build_master_dataset,
    interpolate_with_trend_extrapolation,
    save_figure,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

fig_prefix = "01_"

## Step 1 — IHME mental health prevalence (base dataset)

Source: IHME Global Burden of Disease export (local CSV), already filtered to EU countries at extraction time. `load_ihme_data()` pivots the raw long format (one row per country/year/cause) into a wide format — one column per mental health or neurological cause (e.g. `Depressive disorders`, `Anxiety disorders`, `Schizophrenia`) — since each cause becomes its own predictor for the models in `04_models.ipynb`. The `All causes` aggregate row is dropped, since it is by construction collinear with the sum of the individual causes.

Replace `IHME_CSV_PATH` below with the location of your own IHME export if it differs.

In [ ]:
IHME_CSV_PATH = "../data/raw/IHME-GBD_2023_DATA-b62eec84-1.csv"

df_base = load_ihme_data(IHME_CSV_PATH, min_year=2000)
print(f"IHME base dataset: {df_base.shape[0]} rows, {df_base.shape[1]} columns")
display(df_base.head())

## Step 2 — World Bank socioeconomic indicators (API)

One request per indicator (the World Bank API does not support multi-indicator queries), merged onto `df_base` by `Code` + `Year`. The eight indicators below (`WORLD_BANK_INDICATORS` in `src/config.py`) were chosen to cover the main socioeconomic dimensions associated with suicide risk in the literature:

| Indicator | Rationale |
|---|---|
| GDP per capita | General economic development and financial security |
| Unemployment rate | Direct proxy for economic stress at the individual level |
| Health expenditure (% GDP) | Investment in healthcare systems, including mental health services |
| Population | Scale/control variable |
| Gini index | Income inequality — higher inequality is associated in the literature with greater psychosocial stress, independent of absolute income |
| Urban population (%) | Proxy for urban isolation and access to resources, which can act as either a risk or protective factor |
| Physicians per 1,000 inhabitants | Healthcare system capacity — a lower rate implies lower access to psychiatric and general care |
| Internet users (% of population) | Digital connectivity — can be a risk factor (isolation, harmful content) or a protective one (access to support and information) |

In [ ]:
df_features = fetch_worldbank_indicators(
    df_base,
    eu_countries_iso=EU_COUNTRIES_ISO,
    indicators=WORLD_BANK_INDICATORS,
    region="ALL",
    date_range="2000:2026",
)
print(
    f"\nFeature dataset after World Bank merge: {df_features.shape[0]} rows, {df_features.shape[1]} columns"
)
display(df_features.head())

## Step 3 — WHO suicide rate (target variable, API)

Fetches the age-standardised suicide mortality rate per 100,000 inhabitants from the WHO Global Health Observatory, filtered to both sexes (`SEX_BTSX`) and all age groups (`AGEGROUP_YEARSALL`) to obtain a single comparable rate per country/year, then merged onto `df_features`. Because this is the target variable, `fetch_who_suicide_rates()` raises an error rather than failing silently if the request does not succeed — a silent partial fetch here would corrupt every downstream model without any visible warning.

`build_master_dataset()` then splits the merged data by year into `df_development` (2000–2021, labeled — used for modelling) and `df_real_world` (2022–2023, unlabeled — WHO has not yet published suicide rates for these years, so this set is kept only for future reference/scoring, not for training or evaluation).

In [ ]:
df_who = fetch_who_suicide_rates(indicator=WHO_SUICIDE_INDICATOR)

df_development, df_real_world = build_master_dataset(
    df_features, df_who, development_cutoff_year=2021
)

print(
    f"df_development: {df_development.shape[0]} rows (2000-2021, labeled — used for modelling)"
)
print(
    f"df_real_world:  {df_real_world.shape[0]} rows (2022-2023, unlabeled — reference only)"
)
display(df_development.head())

## Descriptive check and missing values

Before imputing anything, check the overall shape of `df_development` (value ranges, scale differences across features) and identify exactly which columns and rows are missing. Imputing blindly without first inspecting *how* the values are missing risks choosing the wrong method — see the next section.

In [ ]:
print("Master dataset summary statistics")
display(df_development.describe().T)

print("\nMissing values per column:")
remaining_nulls = df_development.isnull().sum()
print(remaining_nulls)

In [ ]:
# Inspect rows with any missing value
missing_mask = df_development.isnull().any(axis=1)
display(df_development[missing_mask])

## Missing value imputation

The columns with missing values (`Gini index` and `Physicians per 100000`, mostly early years with incomplete World Bank coverage for some countries) are missing in consecutive-year blocks per country rather than at scattered individual points, which rules out simple mean/median imputation — that would flatten a real underlying trend. `interpolate_with_trend_extrapolation()` (`src/data_loading.py`) instead:

- **Interior gaps** (missing years between two known values): standard linear interpolation.
- **Boundary gaps** (missing years at the start or end of a country's series, which linear interpolation cannot reach): a linear trend is fit with OLS on that country's own known years and extrapolated, instead of flat-filling with the nearest known value — flat-fill would understate any ongoing trend at the edges of the series.
- If a country has fewer than 2 known points, a trend cannot be estimated, so it falls back to flat-fill for that country.

In [ ]:
features_with_nan = df_development.columns[df_development.isnull().any()].tolist()
print(f"Columns with missing values detected: {features_with_nan}")

# df_development = impute_missing_values(df_development, features_with_nan)
df_development = interpolate_with_trend_extrapolation(df_development, features_with_nan)

print(
    remaining_nulls[remaining_nulls > 0]
    if remaining_nulls.any()
    else "None — all clean."
)

print("\nMissing values per column after imputation:")
print(df_development.isnull().sum())
display(df_development.head())

## Save processed data

Saved so downstream notebooks (`02_eda.ipynb` onward) can load directly without re-running the API calls above.

Saved as Parquet rather than CSV: columnar storage, and — unlike CSV — it preserves each column's dtype exactly across a save/load round-trip, so a downstream notebook never has to re-infer whether a column is an int, a float, or a category. This dataset is small enough that CSV would work fine too; the switch matters more as a habit for datasets that grow, and is documented here so the choice is explicit rather than implicit.

In [ ]:
df_development.to_parquet("../data/processed/df_development.parquet", index=False)
print("Saved: data/processed/df_development.parquet")

df_real_world.to_parquet("../data/processed/df_real_world.parquet", index=False)
print("Saved: data/processed/df_real_world.parquet")